# Parallelism

## Threading

Python provides a threading library that allows the creation of multiple threads that executes within a Python program. It is a bit of code overhead. But we could represent a multithreaded for-loop execution in the following way:

In [1]:
import multiprocessing

nthreads = multiprocessing.cpu_count()
print(f"Number of threads: {nthreads}")

Number of threads: 24


In [2]:
import numpy as np
import threading
import multiprocessing

def worker(arr1, arr2, arr3, chunk):
    """The thread worker."""

    for index in chunk:
        arr3[index] = arr1[index] + arr2[index]

nthreads = multiprocessing.cpu_count()

# 创建三个长度为10000的数组
n = 10000
# print(list(range(n)))
a = np.random.randn(n)
b = np.random.randn(n)
c = np.empty(n, dtype='float64')

def run_with_n_threads(nthreads):
    chunks = np.array_split(range(n), nthreads)
    all_threads = []
    for chunk in chunks:
        # 创建一个线程并告诉它去执行worker(a, b, c, chunk)，参数是(a, b, c, chunk)，并不是立刻启动
        thread = threading.Thread(target=worker, args=(a, b, c, chunk))
        all_threads.append(thread)
        thread.start()
    # 告诉主线程等待所有子线程执行完毕，防止线程没结束就开始计算时间
    for thread in all_threads:
        thread.join()

### Run with 1 thread

Lets run with just 1 thread to see how much time it takes.


In [3]:
%timeit run_with_n_threads(1)

6.23 ms ± 12.8 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### Run with 2 threads

Now lets run with 2 threads to see how much time it takes.

In [4]:
%timeit run_with_n_threads(2)

6.53 ms ± 53.3 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### Run with 4 threads

Okay strange, lets run 4 threads!

In [5]:
%timeit run_with_n_threads(4)

6.99 ms ± 94.4 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


WHAT IS GOING ON? Back to the slides!

## Multiprocessing

Instead of multiple threads we use multiple Python processes, each with its own GIL and memory space.

In [6]:
multiprocessing.get_all_start_methods()

['spawn']

In [7]:
# multiprocessing.set_start_method('fork')
multiprocessing.set_start_method('spawn', force=True)

In [ ]:
# python提供了共享ctypes对象，可以让多个子进程访问同一块共享内存
import ctypes

# Only needed for Jupyter notebooks

def worker(arr1, arr2, arr3, chunk):
    """The thread worker."""

    # Create Numpy arrays from the
    # shared multiprocessing arrays

    # 现在的array是共享内存的array，不能直接用numpy的array操作，所以需要用np.frombuffer()
    # 把共享内存的array转换为numpy array，arr1.get_obj()获得共享内存的array对象，np.frombuffer()
    # 把它转换为numpy array
    arr1_np = np.frombuffer(arr1.get_obj())
    arr2_np = np.frombuffer(arr2.get_obj())
    arr3_np = np.frombuffer(arr3.get_obj())

    for index in chunk:
        arr3_np[index] = arr1_np[index] + arr2_np[index]

nprocesses = multiprocessing.cpu_count()

n = 1000000
# 共享数组
a = multiprocessing.Array(ctypes.c_double, n)
b = multiprocessing.Array(ctypes.c_double, n)
c = multiprocessing.Array(ctypes.c_double, n)


a[:] = np.random.randn(n)
b[:] = np.random.randn(n)



def run_with_n_processes(nprocesses):
    all_processes = []
    chunks = np.array_split(range(n), nprocesses)
    for chunk in chunks:
        process = multiprocessing.Process(target=worker, args=(a, b, c, chunk))
        all_processes.append(process)
        process.start()

    for process in all_processes:
        process.join()

In [ ]:
%timeit run_with_n_processes(1)

In [ ]:
%timeit run_with_n_processes(2)

446 ms ± 69.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
%timeit run_with_n_processes(4)

Ok we are getting somewhere